In [0]:
%pip install Faker

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("nb_lignes", "200", "Nombre de lignes")
nb_lignes = int(dbutils.widgets.get("nb_lignes"))

In [0]:
dbutils.widgets.text("lot", "3", "Lot")
lot = int(dbutils.widgets.get("lot"))
print(f"lot lu = {lot}")

In [0]:
from faker import Faker as f
import json
import random

SEED = 42
f.seed(SEED)
random.seed(SEED)
fake = f("fr_FR")

customers_pool = [
    {
        "customer_id": f"C{i:03d}",
        "name": fake.name(),
        "email": fake.email(),
        "country": fake.country(),
    }
    for i in range(1, 51)
]
print(len(customers_pool))
print(customers_pool[0])

In [0]:
PRODUITS = [
    ("Casque audio", "Électronique"), ("Clavier mécanique", "Électronique"),
    ("Souris sans fil", "Électronique"), ("Écran 27 pouces", "Électronique"),
    ("Montre connectée", "Électronique"), ("Enceinte Bluetooth", "Électronique"),
    ("Webcam HD", "Électronique"),
    ("Lampe de bureau", "Maison"), ("Chaise ergonomique", "Maison"),
    ("Cafetière", "Maison"), ("Bouilloire", "Maison"), ("Plaid en laine", "Maison"),
    ("Tapis de yoga", "Sport"), ("Gourde isotherme", "Sport"),
    ("Ballon de foot", "Sport"), ("Raquette de tennis", "Sport"), ("Sac à dos", "Sport"),
    ("Roman policier", "Livres"), ("Livre de cuisine", "Livres"), ("Carnet de notes", "Livres"),
]
products_pool = [
    {
        "product_id": f"P{i:03d}",
        "name": nom,
        "category": categorie,
        "base_price": round(random.uniform(5, 500), 2),
    }
    for i,(nom, categorie) in enumerate(PRODUITS, start=1)
]

# --- 2. Nouvelle graine : les événements changeront à chaque lot ---
f.seed(SEED + lot)
random.seed(SEED + lot)

print(len(customers_pool), customers_pool[0])
print(len(products_pool), products_pool[0])

In [0]:
from datetime import datetime, timedelta

def make_order(lot, index):
    # 1. Identifiant prévisible : lot + numéro de la commande dans le lot
    order_id = f"O{lot:03d}-{index:04d}"

    # 2. Un client existant, pris au hasard dans le pool
    client = random.choice(customers_pool)

    # 3. Entre 1 et 4 produits différents
    produits = random.sample(products_pool, k=random.randint(1, 4))
    items = [
        {
            "product_id": p["product_id"],
            "quantity": random.randint(1, 3),
            "unit_price": p["base_price"],
        }
        for p in produits
    ]

    # 4. Montant total = somme (quantité × prix unitaire)
    total = round(sum(it["quantity"] * it["unit_price"] for it in items), 2)

    # 5. Heure de la commande : maintenant moins 0 à 60 minutes
    order_ts = (datetime.now() - timedelta(minutes=random.randint(0, 60))).isoformat()

    return {
        "order_id": order_id,
        "customer_id": client["customer_id"],
        "items": items,
        "total_amount": total,
        "order_ts": order_ts,
    }

# Test
print(make_order(lot, 0))
print(make_order(lot, 1))

In [0]:
def make_customer(client, row_status):
    # On part d'une copie du client du pool (sans modifier le pool lui-même)
    message = dict(client)

    PAYS = ["France", "Belgique", "Suisse", "Canada", "Maroc"]

    # Pour une mise à jour : le client change d'email, parfois aussi de pays
    if row_status == "update":
        message["email"] = fake.email()          # ← Faker (graine 42 + lot) sert ici
        if random.random() < 0.3:
            message["country"] = random.choice(PAYS)

    message["row_status"] = row_status
    message["row_time"] = (datetime.now() - timedelta(minutes=random.randint(0, 60))).isoformat()
    return message

# Test
print(make_customer(customers_pool[0], "insert"))
print(make_customer(customers_pool[0], "update"))
print(make_customer(customers_pool[0], "delete"))

In [0]:
def make_product(produit):
    message = dict(produit)

    # Nouveau prix : le prix de base modifié de -20 % à +20 %
    variation = random.uniform(0.8, 1.2)
    message["price"] = round(produit["base_price"] * variation, 2)
    del message["base_price"]

    message["updated_at"] = (datetime.now() - timedelta(minutes=random.randint(0, 60))).isoformat()
    return message

In [0]:
METHODES = ["carte", "paypal", "virement"]

def make_payment(order_id, amount):
    return {
        "payment_id": f"PAY-{order_id}",
        "order_id": order_id,
        "amount": amount,
        "method": random.choice(METHODES),
        "payment_ts": datetime.now().isoformat(),
    }
print(make_product(products_pool[0]))
commande = make_order(lot, 0)
print(make_payment(commande["order_id"], commande["total_amount"]))

In [0]:
# Répartition approximative de nb_lignes entre les topics
n_orders    = int(nb_lignes * 0.35)
n_customers = int(nb_lignes * 0.20)
n_products  = int(nb_lignes * 0.10)
# (les paiements découlent des commandes : environ 35 %)

evenements = []   # liste de paires (topic, message)

# --- COMMANDES ---
orders = [make_order(lot, i) for i in range(n_orders)]
# Piège : ~3 % de montants négatifs
for o in random.sample(orders, k=max(1, int(n_orders * 0.03))):
    o["total_amount"] = -o["total_amount"]
evenements += [("orders", o) for o in orders]

# --- PAIEMENTS ---
# Règle : index % 5 == 0 → payée au lot suivant, == 1 → 2 lots plus tard, sinon tout de suite
for o in orders:
    index = int(o["order_id"].split("-")[1])
    if index % 5 not in (0, 1):
        evenements.append(("payments", make_payment(o["order_id"], abs(o["total_amount"]))))

# Piège : paiements en retard pour les commandes des lots N-1 et N-2
for retard, reste in [(1, 0), (2, 1)]:
    lot_passe = lot - retard
    if lot_passe < 1:          # les lots 1 et 2 n'ont pas (assez) de passé
        continue
    for i in range(n_orders):
        if i % 5 == reste:
            order_id = f"O{lot_passe:03d}-{i:04d}"
            evenements.append(("payments", make_payment(order_id, round(random.uniform(20, 1500), 2))))

# --- CLIENTS ---
if lot == 1:
    # Premier lot : tous les clients s'inscrivent
    clients_msgs = [make_customer(c, "insert") for c in customers_pool]
else:
    # Lots suivants : surtout des mises à jour, quelques suppressions
    clients_msgs = [
        make_customer(random.choice(customers_pool),
                      random.choices(["update", "delete"], weights=[95, 5])[0])
        for _ in range(n_customers)
    ]
    # Piège : 3 clients mis à jour 3 fois dans le même lot, row_time croissant
    for c in random.sample(customers_pool, k=3):
        for k in range(3):
            m = make_customer(c, "update")
            m["row_time"] = (datetime.now() - timedelta(minutes=30 - 10 * k)).isoformat()
            clients_msgs.append(m)

# Piège : ~5 % d'emails vides
for m in random.sample(clients_msgs, k=max(1, int(len(clients_msgs) * 0.05))):
    m["email"] = ""
evenements += [("customers", m) for m in clients_msgs]

# --- PRODUITS ---
evenements += [("products", make_product(random.choice(products_pool))) for _ in range(n_products)]

# Piège : ~5 % de doublons exacts
evenements += random.sample(evenements, k=int(len(evenements) * 0.05))

# Mélange : les topics arrivent entremêlés, comme dans un vrai flux
random.shuffle(evenements)

CLES = {
    "orders":    ("order_id",    "order_ts"),
    "customers": ("customer_id", "row_time"),
    "products":  ("product_id",  "updated_at"),
    "payments":  ("payment_id",  "payment_ts"),
}

lignes = []
for topic, message in evenements:
    champ_cle, champ_ts = CLES[topic]
    lignes.append({
        "topic": topic,
        "key": message[champ_cle],
        "value": json.dumps(message, ensure_ascii=False),
        "timestamp": message[champ_ts],
    })
chemin = f"/Volumes/shopstream/landing/raw/lot_{lot:03d}_{datetime.now():%Y%m%d_%H%M%S}.json"

with open(chemin, "w", encoding="utf-8") as fichier:
    for ligne in lignes:
        fichier.write(json.dumps(ligne, ensure_ascii=False) + "\n")

print(f"Fichier écrit : {chemin}")
print(f"Lot {lot} : {len(evenements)} événements générés")

In [0]:
import os
from pyspark.sql import functions as F

CHEMIN = "/Volumes/shopstream/landing/raw/"
resultats = []

def check(nom, ok):
    print(("✅ " if ok else "❌ ") + nom)
    resultats.append(ok)

# --- Étape 2 : catalogue + 4 schémas ---
schemas = {r[0] for r in spark.sql(
    "SELECT schema_name FROM shopstream.information_schema.schemata").collect()}
check("Catalogue shopstream avec landing, bronze, silver, gold",
      {"landing", "bronze", "silver", "gold"} <= schemas)

# --- Étape 3 : volume landing.raw ---
volumes = {r[0] for r in spark.sql(
    "SELECT volume_name FROM shopstream.information_schema.volumes "
    "WHERE volume_schema = 'landing'").collect()}
check("Volume landing.raw", "raw" in volumes)

# --- Validation : 3 fichiers, un par lot ---
fichiers = [f.name for f in dbutils.fs.ls(CHEMIN) if f.name.endswith(".json")]
check(f"Exactement 3 fichiers (trouvés : {len(fichiers)})", len(fichiers) == 3)
check("Un fichier pour chacun des lots 1, 2, 3",
      sorted(f[:7] for f in fichiers) == ["lot_001", "lot_002", "lot_003"])

# --- Lecture de tous les fichiers ---
df = spark.read.json(CHEMIN).select("*", "_metadata.file_name")

# --- Étape 4 : format de l'enveloppe ---
check("Colonnes topic, key, value, timestamp",
      {"topic", "key", "value", "timestamp"} <= set(df.columns))
check("value est une chaîne brute", dict(df.dtypes)["value"] == "string")

# --- Validation : 4 topics ---
topics = {r[0] for r in df.select("topic").distinct().collect()}
check(f"Les 4 topics présents ({sorted(topics)})",
      topics == {"orders", "customers", "products", "payments"})

# --- Étape 5 : les pièges ---
def champ(nom):
    return F.get_json_object("value", f"$.{nom}")

check("Montants négatifs",
      df.filter((F.col("topic") == "orders") & (champ("total_amount").cast("double") < 0)).count() > 0)
check("Emails vides",
      df.filter((F.col("topic") == "customers") & (champ("email") == "")).count() > 0)
check("Doublons exacts",
      df.groupBy("topic", "key", "value", "timestamp").count().filter("count > 1").count() > 0)
check("Paiements en retard (lot 3 → commandes du lot 1)",
      df.filter((F.col("topic") == "payments")
                & F.col("file_name").startswith("lot_003")
                & champ("order_id").startswith("O001")).count() > 0)
check("Client mis à jour plusieurs fois dans un même lot",
      df.filter((F.col("topic") == "customers") & (champ("row_status") == "update"))
        .groupBy("file_name", "key").count().filter("count >= 3").count() > 0)

# --- Journal ---
check("JOURNAL.md présent dans le Git folder", os.path.exists("JOURNAL.md"))

print()
print("🎉 Phase 0 validée !" if all(resultats)
      else f"⚠️ {resultats.count(False)} point(s) à corriger")

In [0]:
for f in dbutils.fs.ls("/Volumes/shopstream/landing/raw/"):
    print(f.name)